In [1]:
# Part1
# upload the docuemnts
# make the chunking
# make the embedding
# store the embedding to the vector db

# Part2
# Initiate llm model
# Invoke embedding
# merge embedding + user question and pass to llm invoke method


In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.docstore.document import Document
from dotenv import load_dotenv
from pathlib import Path
import pypandoc
load_dotenv()

/Users/pravarsharma/Developer/dev/GPT/06_AI_chain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
EMBEDDING_MODEL = "text-embedding-3-large"
# EMBEDDING_MODEL = "all-MiniLM-L6-v2"
db_name = "vector_db"

In [4]:
filename = "/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.pdf"
file_path = Path(filename)
loader = PyPDFLoader(filename)
documents = loader.load()


In [5]:
import pdfplumber

md_content = ""

with pdfplumber.open(filename) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            md_content += text + "\n\n"

with open(f"{file_path.stem}.md", "w") as f:
    f.write(md_content)

print("Done!")

Done!


In [6]:
new_file_name = "/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md"
documents = ""
with open(new_file_name, "r") as f:
    documents += f.read()
    documents += "\n\n"
documents = [Document(page_content=documents, metadata={"source": new_file_name})]


In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)
len(chunks)
chunks[0]


Document(metadata={'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md'}, page_content='https://www.linkedin.com/in/pravar-sharma-3410a199/\nPravar Sharma\nhttps://github.com/pravar1919\npravar.sharma@gmail.com\n+91 9414472171\nProfessional Summary\nInnovative Senior Backend & Cloud Engineer with 5 years of experience building scalable, secure\nenterprise solutions. Expert in Python, FastAPI, Django, Microservices, and AWS/GCP. Hands-on')

In [8]:
chunks[1]

Document(metadata={'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md'}, page_content='enterprise solutions. Expert in Python, FastAPI, Django, Microservices, and AWS/GCP. Hands-on\nexperience in RAG-based AI systems, vector databases, and LLM-powered application\ndevelopment. Strong background in CI/CD, containerization, event-driven architecture, and\nperformance optimization. Passionate about designing automation, improving developer')

In [9]:
chunks[2]

Document(metadata={'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md'}, page_content='performance optimization. Passionate about designing automation, improving developer\nexperience, and delivering business value through technology.\nWork Experience\nSenior Engineer - Cloud Services & Software\nLTIMindtree | Sept 2023 - Present\nDeveloped new features and enhanced existing functionalities to meet evolving client\nrequirements.')

In [10]:
# embeddings = HuggingFaceEmbeddings(model=EMBEDDING_MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 21 documents


In [11]:
# Part 2
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [26]:
# llm = ChatOpenAI(temperature=0, model="gpt-5-nano")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.6)

In [ ]:
retriver = vectorstore.as_retriever()

In [33]:
retriver.invoke("what is the name?", k=10)

[Document(id='e918ee9a-407b-4b72-9725-4f3f83d986f0', metadata={'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md'}, page_content='Tech Stack: FastAPI, Python, LangChain, FAISS, OpenAI GPT-4.1, OpenAI Embeddings, PyMuPDF,\nDocker, uv\nAI-Powered Natural Language SQL Query Assistant (FastAPI + LangChain + OpenAI + PostgreSQL)\nConverts business questions (NL) into SQL queries using OpenAI GPT-4.\nExecutes SQL queries safely, retrieving validated results.\nUses RAG-style schema awareness for accuracy & grounding.'),
 Document(id='a9efe5db-f8b5-4837-8e62-7f329726b1bf', metadata={'source': '/Users/pravarsharma/Developer/dev/GPT/03_rag_read_from_pdf/Pravar-Sharma.md'}, page_content='ThehouseofMUSH | eCommerce Platform\nDeveloped a full-featured eCommerce platform with Django and Tailwind CSS.\nIntegrated secure payment gateway (Razorpay), analytics, and user tracking for better sales insights.\nImplemented scalable architecture, performance optimizations,

In [17]:
llm.invoke("what is the name?")

AIMessage(content="This conversation has just started, so I'm not aware of any specific topic or context you're referring to. Could you please provide more information or clarify what you're asking about?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 40, 'total_tokens': 77, 'completion_time': 0.058309279, 'completion_tokens_details': None, 'prompt_time': 0.002909862, 'prompt_tokens_details': None, 'queue_time': 0.049819587, 'total_time': 0.061219141}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_ff2b098aaf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--0dedb2db-1e75-44ab-a48d-70c07ad450b9-0', usage_metadata={'input_tokens': 40, 'output_tokens': 37, 'total_tokens': 77})

In [29]:
# Now will combine retriver output + the user input and fed that to the llm.
system_prompt_template = """
    You are a knowledgeable, friendly assistant representing the resume of a candiate.
    You are chatting with a user about the resume of a candiate.
    If relevant, use the given context to answer any question.
    If you don't know the answer, say so.
    You don't have to answer anything beyond the context, simply say so.
    At the very end of your answer, aks user some of the follow up questions like 
    "would you also like to know about ...." something like that
    Context:
    {context}
"""

In [32]:
def answer_question(question, history):

    # Retrieve docs (RAG)
    docs = retriver.invoke(question, k=10)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Start message list with system prompt
    messages = [
        SystemMessage(content=system_prompt_template.format(context=context))
    ]

    # Convert Gradio ChatInterface history -> LLM format
    for msg in history:
        if msg["role"] == "user":
            messages.append(HumanMessage(content=msg["content"]))
        else:
            messages.append(AIMessage(content=msg["content"]))

    # Add latest question
    messages.append(HumanMessage(content=question))

    # LLM response
    response = llm.invoke(messages)

    # Return ONLY reply string (ChatInterface updates history itself)
    return response.content


In [20]:
answer_question("is the candidate is suitable for generative AI profile?", [])

'Based on the provided resume, the candidate has hands-on experience in LLM-powered application development, specifically mentioning OpenAI GPT-4.1 and OpenAI Embeddings. They also have experience with RAG-based AI systems, which involves integrating language models with retrieval systems.\n\nAdditionally, the candidate has worked on a project that involves using a Retrieval-Augmented Generation (RAG) pipeline to evaluate candidate-role fit using semantic similarity between resumes and job descriptions. This suggests that they have a good understanding of how to apply generative AI techniques to real-world problems.\n\nHowever, to determine if the candidate is a good fit for a generative AI profile, it would be helpful to know more about the specific requirements of the role, such as:\n\n* The type of generative AI techniques being used (e.g. text-to-image, text-to-text, etc.)\n* The level of experience required with specific tools and technologies (e.g. Hugging Face, LangChain, etc.)\

In [21]:
import gradio as gr

In [34]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
